# Adult Income Classification
**Objetivo:** Predecir si una persona gana `>50K` al año (clasificación binaria).

Dataset: UCI Adult (32,561 filas, sin header).

---
## Fase 0 — Setup

In [1]:
# !pip install -q pandas numpy scikit-learn matplotlib seaborn xgboost optuna hyperopt umap-learn joblib

In [2]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from xgboost import XGBClassifier
print("Librerías importadas correctamente.")

Librerías importadas correctamente.


In [3]:
RANDOM_STATE = 42
DATA_PATH = r"C:\Users\USUARIO1\Documents\IA\data\adult.csv"
TARGET_COL = "income"
MODEL_OUTPUT_DIR = r"C:\Users\USUARIO1\Documents\IA\Adult\models"
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
COLUMN_NAMES = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]
print("Constantes OK")

Constantes OK


In [4]:
df = pd.read_csv(DATA_PATH, names=COLUMN_NAMES, skipinitialspace=True)
assert df.shape == (32561, 15)
print("[OK] Fase 0: shape=(32561, 15)")

[OK] Fase 0: shape=(32561, 15)


---
## Fase 1 — Limpieza de datos

In [5]:
df = df.replace("?", np.nan)
for col in ["workclass", "occupation", "native_country"]:
    df[col] = df[col].fillna(df[col].mode()[0])
df[TARGET_COL] = df[TARGET_COL].str.replace(".", "", regex=False).str.strip()
df[TARGET_COL] = df[TARGET_COL].map({"<=50K": 0, ">50K": 1})
df = df.drop(columns=["fnlwgt"])
y = df[TARGET_COL].values
X_raw = df.drop(columns=[TARGET_COL])
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
preprocessor = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
X = preprocessor.fit_transform(X_raw)
ohe = preprocessor.named_transformers_["cat"]
feature_names = num_cols + ohe.get_feature_names_out(cat_cols).tolist()
df_clean = pd.DataFrame(X, columns=feature_names)
df_clean[TARGET_COL] = y
assert np.isnan(X).sum() == 0
assert set(np.unique(y)) == {0, 1}
print(f"[OK] Fase 1: X.shape={X.shape}, NaN=0, clases={{0,1}}")

[OK] Fase 1: X.shape=(32561, 104), NaN=0, clases={0,1}


---
## Fase 2 — EDA

In [6]:
print("=== Estadísticas descriptivas ===")
print(df_clean[num_cols].describe().T.round(2).to_string())

=== Estadísticas descriptivas ===
                  count     mean      std   min   25%   50%   75%      max
age             32561.0    38.58    13.64  17.0  28.0  37.0  48.0     90.0
education_num   32561.0    10.08     2.57   1.0   9.0  10.0  12.0     16.0
capital_gain    32561.0  1077.65  7385.29   0.0   0.0   0.0   0.0  99999.0
capital_loss    32561.0    87.30   402.96   0.0   0.0   0.0   0.0   4356.0
hours_per_week  32561.0    40.44    12.35   1.0  40.0  40.0  45.0     99.0


In [7]:
class_counts = pd.Series(y).value_counts().rename({0: "<=50K", 1: ">50K"})
print("Distribución target:", class_counts.to_dict())
print("Proporción:", (class_counts / len(y)).round(4).to_dict())

Distribución target: {'<=50K': 24720, '>50K': 7841}
Proporción: {'<=50K': 0.7592, '>50K': 0.2408}


In [8]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, num_cols):
    ax.hist(df_clean[col], bins=30, color="steelblue", edgecolor="black")
    ax.set_title(col)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, "eda_histogramas.png"), dpi=100, bbox_inches="tight")
plt.close()
fig, ax = plt.subplots(figsize=(6, 4))
counts = pd.Series(y).value_counts().sort_index()
ax.bar(["<=50K (0)", ">50K (1)"], counts.values, color=["steelblue", "tomato"], edgecolor="black")
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, str(v), ha="center", fontsize=11)
ax.set_title("Distribución del target")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, "eda_target.png"), dpi=100, bbox_inches="tight")
plt.close()
corr_with_target = df_clean.corr()[TARGET_COL].drop(TARGET_COL).abs().sort_values(ascending=False)
top20_cols = corr_with_target.head(20).index.tolist()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(df_clean[top20_cols + [TARGET_COL]].corr(), annot=False, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación: top 20 features vs income")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, "eda_heatmap.png"), dpi=100, bbox_inches="tight")
plt.close()
print("[OK] Fase 2: 3 figuras guardadas.")

[OK] Fase 2: 3 figuras guardadas.


---
## Fase 3 — Baseline Naive Bayes

In [9]:

# Split 80/20 exclusivo para baseline (sobre X_raw, no sobre X OHE)
X_train_raw, X_test_raw, y_train_full, y_test_nb = train_test_split(
    X_raw, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train_raw.shape[0]} | Test: {X_test_raw.shape[0]}")
print(f"Train pos%: {(y_train_full==1).mean():.4f} | Test pos%: {(y_test_nb==1).mean():.4f}")


Train: 26048 | Test: 6513
Train pos%: 0.2408 | Test pos%: 0.2407


In [10]:

# Pipeline NB: OrdinalEncoder para categóricas + StandardScaler para numéricas
# GaussianNB asume distribución gaussiana: OHE binario viola ese supuesto.
# OrdinalEncoder produce valores enteros continuos, compatibles con GNB.
from sklearn.preprocessing import OrdinalEncoder

nb_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
])

X_train_nb = nb_preprocessor.fit_transform(X_train_raw)
X_test_nb_s = nb_preprocessor.transform(X_test_raw)
print(f"Shape tras preprocesado NB: train={X_train_nb.shape}, test={X_test_nb_s.shape}")
print(f"({len(num_cols)} cols escaladas + {len(cat_cols)} cols ordinales)")


Shape tras preprocesado NB: train=(26048, 13), test=(6513, 13)
(5 cols escaladas + 8 cols ordinales)


In [11]:
# Entrenar GaussianNB
nb = GaussianNB()
nb.fit(X_train_nb, y_train_full)
y_pred_nb = nb.predict(X_test_nb_s)

results_baseline = {
    "model":     "GaussianNB",
    "accuracy":  accuracy_score(y_test_nb, y_pred_nb),
    "precision": precision_score(y_test_nb, y_pred_nb),
    "recall":    recall_score(y_test_nb, y_pred_nb),
    "f1":        f1_score(y_test_nb, y_pred_nb),
}
print("\n=== Baseline GaussianNB ===")
for k, v in results_baseline.items():
    print(f"  {k:12s}: {v if isinstance(v, str) else f'{v:.4f}'}")


=== Baseline GaussianNB ===
  model       : GaussianNB
  accuracy    : 0.8099
  precision   : 0.7078
  recall      : 0.3584
  f1          : 0.4759


In [12]:
print("\nClassification Report:")
print(classification_report(y_test_nb, y_pred_nb, target_names=["<=50K", ">50K"]))

cm = confusion_matrix(y_test_nb, y_pred_nb)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["<=50K", ">50K"], yticklabels=["<=50K", ">50K"], ax=ax)
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
ax.set_title("Confusion Matrix — GaussianNB")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, "cm_gaussiannb.png"), dpi=100, bbox_inches="tight")
plt.close()
print("Matriz de confusión guardada.")


Classification Report:
              precision    recall  f1-score   support

       <=50K       0.82      0.95      0.88      4945
        >50K       0.71      0.36      0.48      1568

    accuracy                           0.81      6513
   macro avg       0.77      0.66      0.68      6513
weighted avg       0.80      0.81      0.79      6513



Matriz de confusión guardada.

In [13]:
# Validacion Fase 3
required_keys = {"model", "accuracy", "precision", "recall", "f1"}
assert required_keys == set(results_baseline.keys()), "Faltan claves en results_baseline"
assert results_baseline["accuracy"] > 0.70, f"Accuracy demasiado baja: {results_baseline['accuracy']:.4f}"
print("[OK] Fase 3 validada:")
print(f"  accuracy={results_baseline['accuracy']:.4f} | precision={results_baseline['precision']:.4f} | recall={results_baseline['recall']:.4f} | f1={results_baseline['f1']:.4f}")

[OK] Fase 3 validada:
  accuracy=0.8099 | precision=0.7078 | recall=0.3584 | f1=0.4759


---
## Fase 4 — Modelos avanzados

In [14]:

# Split estratificado 60/20/20 sobre X (OHE) e y
# Paso 1: separar 80% (train+val) y 20% test
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
# Paso 2: del 80% separar 75% train / 25% val → 60% y 20% del total
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=RANDOM_STATE
)
print(f"train : {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"val   : {X_val.shape[0]}  ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"test  : {X_test.shape[0]}  ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Pos% -> train={y_train.mean():.4f} | val={y_val.mean():.4f} | test={y_test.mean():.4f}")


train : 19536 (60.0%)


val   : 6512  (20.0%)
test  : 6513  (20.0%)


Pos% -> train=0.2408 | val=0.2408 | test=0.2407


In [15]:

# Función de evaluación reutilizable en todas las fases
def evaluate(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    return {
        "accuracy":  round(accuracy_score(y_eval, y_pred), 4),
        "precision": round(precision_score(y_eval, y_pred), 4),
        "recall":    round(recall_score(y_eval, y_pred), 4),
        "f1":        round(f1_score(y_eval, y_pred), 4),
    }

print("Función evaluate() definida.")


Función evaluate() definida.


In [16]:

# Definir 4 modelos con parámetros por defecto
models = {
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "AdaBoost":     AdaBoostClassifier(random_state=RANDOM_STATE),
    "XGBoost":      XGBClassifier(
                        random_state=RANDOM_STATE,
                        eval_metric="logloss",
                        n_jobs=-1,
                        verbosity=0,
                    ),
}

results_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    val_metrics  = evaluate(model, X_val,  y_val)
    test_metrics = evaluate(model, X_test, y_test)
    results_models[name] = {
        "val":       val_metrics,
        "test":      test_metrics,
        "estimator": model,
    }
    print(f"{name:15s} | val  F1={val_metrics['f1']:.4f}  acc={val_metrics['accuracy']:.4f}"
          f" | test F1={test_metrics['f1']:.4f}  acc={test_metrics['accuracy']:.4f}")


DecisionTree    | val  F1=0.6145  acc=0.8176 | test F1=0.6303  acc=0.8240

RandomForest    | val  F1=0.6588  acc=0.8446 | test F1=0.6723  acc=0.8501


AdaBoost        | val  F1=0.6484  acc=0.8503 | test F1=0.6636  acc=0.8552


XGBoost         | val  F1=0.7101  acc=0.8696 | test F1=0.7181  acc=0.8719


In [17]:

# Tabla comparativa val vs test por modelo
print("\n=== Tabla comparativa — test ===")
print(f"{'Modelo':15s} | {'accuracy':>8} | {'precision':>9} | {'recall':>6} | {'f1':>6}")
print("-" * 55)
for name, res in results_models.items():
    m = res["test"]
    print(f"{name:15s} | {m['accuracy']:>8.4f} | {m['precision']:>9.4f} | {m['recall']:>6.4f} | {m['f1']:>6.4f}")

# Matrices de confusión (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, res) in zip(axes.flat, results_models.items()):
    y_pred = res["estimator"].predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["<=50K", ">50K"], yticklabels=["<=50K", ">50K"], ax=ax)
    ax.set_title(f"{name}  (F1={res['test']['f1']:.4f})")
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
plt.suptitle("Confusion Matrices — Modelos avanzados (test)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, "cm_advanced_models.png"), dpi=100, bbox_inches="tight")
plt.close()
print("Matrices de confusión guardadas.")



=== Tabla comparativa — test ===
Modelo          | accuracy | precision | recall |     f1
-------------------------------------------------------
DecisionTree    |   0.8240 |    0.6377 | 0.6231 | 0.6303
RandomForest    |   0.8501 |    0.7099 | 0.6384 | 0.6723
AdaBoost        |   0.8552 |    0.7530 | 0.5931 | 0.6636
XGBoost         |   0.8719 |    0.7640 | 0.6773 | 0.7181


Matrices de confusión guardadas.


In [18]:

# Validacion Fase 4
expected_models = {"DecisionTree", "RandomForest", "AdaBoost", "XGBoost"}
assert set(results_models.keys()) == expected_models, "Faltan modelos en results_models"
for name, res in results_models.items():
    assert "val"       in res, f"{name}: falta clave 'val'"
    assert "test"      in res, f"{name}: falta clave 'test'"
    assert "estimator" in res, f"{name}: falta clave 'estimator'"
    assert res["test"]["f1"] > 0.60, f"{name}: F1 test demasiado bajo ({res['test']['f1']})"
print("[OK] Fase 4 validada: 4 modelos entrenados y evaluados en val+test.")
best_f4 = max(results_models, key=lambda k: results_models[k]["test"]["f1"])
print(f"  Mejor en test: {best_f4}  F1={results_models[best_f4]['test']['f1']:.4f}")


[OK] Fase 4 validada: 4 modelos entrenados y evaluados en val+test.
  Mejor en test: XGBoost  F1=0.7181


---
## Fase 5 — Optimización de hiperparámetros

In [19]:

results_hpo = {"GridSearch": {}, "Random": {}, "Optuna": {}, "Hyperopt": {}, "NNI": {}}

def register_hpo(method, model_name, best_params, estimator):
    """Entrena el estimator si no está entrenado, evalúa en val y test, registra."""
    estimator.fit(X_train, y_train)
    results_hpo[method][model_name] = {
        "best_params": best_params,
        "val":         evaluate(estimator, X_val,  y_val),
        "test":        evaluate(estimator, X_test, y_test),
        "estimator":   estimator,
    }
    v = results_hpo[method][model_name]["val"]["f1"]
    t = results_hpo[method][model_name]["test"]["f1"]
    print(f"  [{method}] {model_name:12s} -> val F1={v:.4f} | test F1={t:.4f}")

print("Estructura results_hpo inicializada. Función register_hpo() definida.")


Estructura results_hpo inicializada. Función register_hpo() definida.


In [20]:

# ── 5.1  GridSearchCV ────────────────────────────────────────────────────────
print("=== 5.1  GridSearchCV ===")

rf_grid = {
    "n_estimators":      [100, 200],
    "max_depth":         [None, 10, 20],
    "min_samples_split": [2, 5],
}
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=rf_grid, scoring="f1", cv=3, n_jobs=-1
)
gs_rf.fit(X_train, y_train)
register_hpo("GridSearch", "RandomForest", gs_rf.best_params_, gs_rf.best_estimator_)

xgb_grid = {
    "n_estimators": [100, 200],
    "max_depth":    [3, 6],
    "learning_rate":[0.05, 0.1],
}
gs_xgb = GridSearchCV(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1, verbosity=0),
    param_grid=xgb_grid, scoring="f1", cv=3, n_jobs=-1
)
gs_xgb.fit(X_train, y_train)
register_hpo("GridSearch", "XGBoost", gs_xgb.best_params_, gs_xgb.best_estimator_)

print("  RF  best params:", gs_rf.best_params_)
print("  XGB best params:", gs_xgb.best_params_)


=== 5.1  GridSearchCV ===


  [GridSearch] RandomForest -> val F1=0.6818 | test F1=0.6984


  [GridSearch] XGBoost      -> val F1=0.7107 | test F1=0.7289
  RF  best params: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}
  XGB best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}


In [21]:

# ── 5.2  RandomizedSearchCV ──────────────────────────────────────────────────
from scipy.stats import randint, uniform
print("=== 5.2  RandomizedSearchCV ===")

rf_dist = {
    "n_estimators":      randint(100, 400),
    "max_depth":         [None, 10, 20, 30],
    "min_samples_split": randint(2, 10),
    "max_features":      ["sqrt", "log2"],
}
rs_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_dist, n_iter=15, scoring="f1",
    cv=3, n_jobs=-1, random_state=RANDOM_STATE
)
rs_rf.fit(X_train, y_train)
register_hpo("Random", "RandomForest", rs_rf.best_params_, rs_rf.best_estimator_)

xgb_dist = {
    "n_estimators":     randint(100, 400),
    "max_depth":        randint(3, 10),
    "learning_rate":    uniform(0.01, 0.3),
    "subsample":        uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
}
rs_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1, verbosity=0),
    param_distributions=xgb_dist, n_iter=15, scoring="f1",
    cv=3, n_jobs=-1, random_state=RANDOM_STATE
)
rs_xgb.fit(X_train, y_train)
register_hpo("Random", "XGBoost", rs_xgb.best_params_, rs_xgb.best_estimator_)

print("  RF  best params:", rs_rf.best_params_)
print("  XGB best params:", rs_xgb.best_params_)


=== 5.2  RandomizedSearchCV ===


  [Random] RandomForest -> val F1=0.6802 | test F1=0.6920


  [Random] XGBoost      -> val F1=0.7091 | test F1=0.7141
  RF  best params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 9, 'n_estimators': 199}
  XGB best params: {'colsample_bytree': np.float64(0.9637281608315128), 'learning_rate': np.float64(0.08763399448000507), 'max_depth': 6, 'n_estimators': 101, 'subsample': np.float64(0.7700623497964979)}


In [22]:

# ── 5.3  Optuna (TPE Bayesiano) ───────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("=== 5.3  Optuna (TPE) ===")

def objective_rf_optuna(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 400),
        "max_depth":         trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2"]),
    }
    m = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train, y_train)
    return f1_score(y_val, m.predict(X_val))

study_rf = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_rf.optimize(objective_rf_optuna, n_trials=20, show_progress_bar=False)
best_rf_opt = RandomForestClassifier(**study_rf.best_params, random_state=RANDOM_STATE, n_jobs=-1)
register_hpo("Optuna", "RandomForest", study_rf.best_params, best_rf_opt)

def objective_xgb_optuna(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 400),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    m = XGBClassifier(**params, random_state=RANDOM_STATE,
                      eval_metric="logloss", n_jobs=-1, verbosity=0)
    m.fit(X_train, y_train)
    return f1_score(y_val, m.predict(X_val))

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_xgb.optimize(objective_xgb_optuna, n_trials=20, show_progress_bar=False)
best_xgb_opt = XGBClassifier(**study_xgb.best_params, random_state=RANDOM_STATE,
                              eval_metric="logloss", n_jobs=-1, verbosity=0)
register_hpo("Optuna", "XGBoost", study_xgb.best_params, best_xgb_opt)

print("  RF  best params:", study_rf.best_params)
print("  XGB best params:", study_xgb.best_params)


=== 5.3  Optuna (TPE) ===


  [Optuna] RandomForest -> val F1=0.6867 | test F1=0.6978


  [Optuna] XGBoost      -> val F1=0.7137 | test F1=0.7282
  RF  best params: {'n_estimators': 106, 'max_depth': 30, 'min_samples_split': 9, 'max_features': 'sqrt'}
  XGB best params: {'n_estimators': 389, 'max_depth': 8, 'learning_rate': 0.03001215871999436, 'subsample': 0.6071847502459279, 'colsample_bytree': 0.8607466203112714}


In [23]:

# ── 5.4  Hyperopt (TPE) ───────────────────────────────────────────────────────
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
print("=== 5.4  Hyperopt ===")

# Espacios con hp.choice — guardar listas para decodificar índices
_rf_choices = {
    "n_estimators":      [100, 200, 300, 400],
    "max_depth":         [5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "max_features":      ["sqrt", "log2"],
}
space_rf_hy = {k: hp.choice(k, v) for k, v in _rf_choices.items()}

def _hy_rf(params):
    m = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train, y_train)
    return {"loss": -f1_score(y_val, m.predict(X_val)), "status": STATUS_OK}

trials_rf_hy = Trials()
best_idx_rf = fmin(_hy_rf, space_rf_hy, algo=tpe.suggest, max_evals=20,
                   trials=trials_rf_hy, rstate=np.random.default_rng(RANDOM_STATE),
                   verbose=False)
best_params_rf_hy = {k: _rf_choices[k][int(v)] for k, v in best_idx_rf.items()}
best_rf_hy = RandomForestClassifier(**best_params_rf_hy, random_state=RANDOM_STATE, n_jobs=-1)
register_hpo("Hyperopt", "RandomForest", best_params_rf_hy, best_rf_hy)

_xgb_choices = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth":    [3, 5, 7, 10],
}
space_xgb_hy = {
    "n_estimators":     hp.choice("n_estimators", _xgb_choices["n_estimators"]),
    "max_depth":        hp.choice("max_depth",    _xgb_choices["max_depth"]),
    "learning_rate":    hp.uniform("learning_rate",    0.01, 0.3),
    "subsample":        hp.uniform("subsample",        0.6,  1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6,  1.0),
}

def _hy_xgb(params):
    m = XGBClassifier(**params, random_state=RANDOM_STATE,
                      eval_metric="logloss", n_jobs=-1, verbosity=0)
    m.fit(X_train, y_train)
    return {"loss": -f1_score(y_val, m.predict(X_val)), "status": STATUS_OK}

trials_xgb_hy = Trials()
best_idx_xgb = fmin(_hy_xgb, space_xgb_hy, algo=tpe.suggest, max_evals=20,
                    trials=trials_xgb_hy, rstate=np.random.default_rng(RANDOM_STATE),
                    verbose=False)
best_params_xgb_hy = {
    "n_estimators":     _xgb_choices["n_estimators"][int(best_idx_xgb["n_estimators"])],
    "max_depth":        _xgb_choices["max_depth"][int(best_idx_xgb["max_depth"])],
    "learning_rate":    float(best_idx_xgb["learning_rate"]),
    "subsample":        float(best_idx_xgb["subsample"]),
    "colsample_bytree": float(best_idx_xgb["colsample_bytree"]),
}
best_xgb_hy = XGBClassifier(**best_params_xgb_hy, random_state=RANDOM_STATE,
                             eval_metric="logloss", n_jobs=-1, verbosity=0)
register_hpo("Hyperopt", "XGBoost", best_params_xgb_hy, best_xgb_hy)

print("  RF  best params:", best_params_rf_hy)
print("  XGB best params:", best_params_xgb_hy)


=== 5.4  Hyperopt ===


  [Hyperopt] RandomForest -> val F1=0.6825 | test F1=0.6947


  [Hyperopt] XGBoost      -> val F1=0.7118 | test F1=0.7223
  RF  best params: {'max_depth': 30, 'max_features': 'sqrt', 'min_samples_split': 10, 'n_estimators': 300}
  XGB best params: {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.11169815504926653, 'subsample': 0.7299796125452439, 'colsample_bytree': 0.9906101216648739}


In [24]:

# ── 5.5  NNI — ejemplo funcional mínimo (local, sin launcher) ────────────────
# En un experimento NNI real se definen search_space.json, trial.py y config.yml.
# Aquí simulamos el loop de NNI con random sampling sobre el mismo espacio.
print("=== 5.5  NNI (simulación local) ===")

nni_space_rf = {
    "n_estimators":      [100, 200, 300],
    "max_depth":         [10, 20, 30],
    "min_samples_split": [2, 5, 10],
}
nni_space_xgb = {
    "n_estimators":  [100, 200, 300],
    "max_depth":     [3, 5, 7],
    "learning_rate": [0.05, 0.1, 0.2],
}

rng_nni = np.random.default_rng(RANDOM_STATE)

# RF — 10 trials
best_score, best_params_nni_rf, best_model_nni_rf = -1, None, None
for _ in range(10):
    params = {k: rng_nni.choice(v).item() for k, v in nni_space_rf.items()}
    m = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train, y_train)
    score = f1_score(y_val, m.predict(X_val))
    if score > best_score:
        best_score, best_params_nni_rf, best_model_nni_rf = score, params, m

register_hpo("NNI", "RandomForest", best_params_nni_rf, best_model_nni_rf)

# XGBoost — 10 trials
best_score, best_params_nni_xgb, best_model_nni_xgb = -1, None, None
for _ in range(10):
    params = {k: rng_nni.choice(v).item() for k, v in nni_space_xgb.items()}
    m = XGBClassifier(**params, random_state=RANDOM_STATE,
                      eval_metric="logloss", n_jobs=-1, verbosity=0)
    m.fit(X_train, y_train)
    score = f1_score(y_val, m.predict(X_val))
    if score > best_score:
        best_score, best_params_nni_xgb, best_model_nni_xgb = score, params, m

register_hpo("NNI", "XGBoost", best_params_nni_xgb, best_model_nni_xgb)

print("  RF  best params:", best_params_nni_rf)
print("  XGB best params:", best_params_nni_xgb)


=== 5.5  NNI (simulación local) ===


  [NNI] RandomForest -> val F1=0.6834 | test F1=0.6990


  [NNI] XGBoost      -> val F1=0.7127 | test F1=0.7274
  RF  best params: {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 10}
  XGB best params: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.2}


In [25]:

# Resumen HPO y validación Fase 5
print("\n=== Resumen HPO — test F1 ===")
print(f"{'Método':12s} | {'RF F1':>8} | {'XGB F1':>8}")
print("-" * 35)
for method in ["GridSearch", "Random", "Optuna", "Hyperopt", "NNI"]:
    rf_f1  = results_hpo[method]["RandomForest"]["test"]["f1"]
    xgb_f1 = results_hpo[method]["XGBoost"]["test"]["f1"]
    print(f"{method:12s} | {rf_f1:>8.4f} | {xgb_f1:>8.4f}")

# Validación
assert len(results_hpo) == 5
for method in results_hpo:
    for model_name in ["RandomForest", "XGBoost"]:
        assert model_name in results_hpo[method], f"Falta {model_name} en {method}"
        entry = results_hpo[method][model_name]
        assert {"best_params","val","test","estimator"} == set(entry.keys())
print("\n[OK] Fase 5 validada: 5 métodos × 2 modelos = 10 entradas en results_hpo.")

best_overall = max(
    [(m, mn, results_hpo[m][mn]["test"]["f1"])
     for m in results_hpo for mn in results_hpo[m]],
    key=lambda x: x[2]
)
print(f"  Mejor HPO global: [{best_overall[0]}] {best_overall[1]}  F1={best_overall[2]:.4f}")



=== Resumen HPO — test F1 ===
Método       |    RF F1 |   XGB F1
-----------------------------------
GridSearch   |   0.6984 |   0.7289
Random       |   0.6920 |   0.7141
Optuna       |   0.6978 |   0.7282
Hyperopt     |   0.6947 |   0.7223
NNI          |   0.6990 |   0.7274

[OK] Fase 5 validada: 5 métodos × 2 modelos = 10 entradas en results_hpo.
  Mejor HPO global: [GridSearch] XGBoost  F1=0.7289


---
## Fase 6 — Reducción de dimensionalidad

In [26]:
import umap

print("=== Fase 6 — Reduccion de dimensionalidad ===")

scaler_dr = StandardScaler()
X_train_s = scaler_dr.fit_transform(X_train)
X_test_s  = scaler_dr.transform(X_test)

results_dimred = {}

def quick_rf_eval(Xtr, ytr, Xte, yte):
    m = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(Xtr, ytr)
    return evaluate(m, Xte, yte)


=== Fase 6 — Reduccion de dimensionalidad ===


In [27]:
# ---- PCA 95% ----
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)
print(f"PCA -> n_components={pca.n_components_}, var. explicada={pca.explained_variance_ratio_.sum():.4f}")
results_dimred["PCA"] = {
    "n_components": int(pca.n_components_),
    "explained_variance": float(pca.explained_variance_ratio_.sum()),
    "metrics": quick_rf_eval(X_train_pca, y_train, X_test_pca, y_test),
}
plt.figure(figsize=(7, 5))
plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, s=4, cmap="coolwarm", alpha=0.5)
plt.title("PCA (2 primeras componentes)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()
print(f"PCA RF F1={results_dimred['PCA']['metrics']['f1']:.4f}")


PCA -> n_components=86, var. explicada=0.9571


PCA RF F1=0.6114


In [28]:
# ---- LDA ----
lda = LinearDiscriminantAnalysis(n_components=1)
X_train_lda = lda.fit_transform(X_train_s, y_train)
X_test_lda  = lda.transform(X_test_s)
results_dimred["LDA"] = {
    "n_components": 1,
    "metrics": quick_rf_eval(X_train_lda, y_train, X_test_lda, y_test),
}
plt.figure(figsize=(7, 4))
plt.hist(X_train_lda[y_train == 0], bins=50, alpha=0.6, label="<=50K")
plt.hist(X_train_lda[y_train == 1], bins=50, alpha=0.6, label=">50K")
plt.title("LDA (1 componente)"); plt.legend(); plt.show()
print(f"LDA RF F1={results_dimred['LDA']['metrics']['f1']:.4f}")


LDA RF F1=0.5421


In [29]:
# ---- t-SNE (solo visualizacion, subsample 3000) ----
sub_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_train_s), size=3000, replace=False)
X_tsne = TSNE(n_components=2, random_state=RANDOM_STATE, init="pca", learning_rate="auto").fit_transform(X_train_s[sub_idx])
plt.figure(figsize=(7, 5))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_train[sub_idx], s=5, cmap="coolwarm", alpha=0.7)
plt.title("t-SNE (subsample 3000)")
plt.show()
results_dimred["tSNE"] = {"n_components": 2, "visualization_only": True}
print("t-SNE: solo visualizacion (sin metricas de modelo)")


t-SNE: solo visualizacion (sin metricas de modelo)


In [30]:
# ---- UMAP ----
umap_model = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_jobs=-1)
X_train_umap = umap_model.fit_transform(X_train_s)
X_test_umap  = umap_model.transform(X_test_s)
results_dimred["UMAP"] = {
    "n_components": 2,
    "metrics": quick_rf_eval(X_train_umap, y_train, X_test_umap, y_test),
}
plt.figure(figsize=(7, 5))
plt.scatter(X_train_umap[:, 0], X_train_umap[:, 1], c=y_train, s=4, cmap="coolwarm", alpha=0.5)
plt.title("UMAP (2D)")
plt.show()
umap_f1 = results_dimred['UMAP']['metrics']['f1']
print(f"UMAP RF F1={umap_f1:.4f}")

# Validacion Fase 6
assert set(results_dimred.keys()) == {"PCA", "LDA", "tSNE", "UMAP"}, "Faltan tecnicas en results_dimred"
assert results_dimred["PCA"]["n_components"] < X_train.shape[1], "PCA no redujo dimensiones"
print("[OK] Fase 6 validada.")


UMAP RF F1=0.5615
[OK] Fase 6 validada.


---
## Fase 7 — Comparación final

In [31]:
print("=== Fase 7 — Comparacion final ===")

rows = []

# Baseline
rows.append({
    "method": "Baseline",
    "model": results_baseline["model"],
    **{k: results_baseline[k] for k in ["accuracy", "precision", "recall", "f1"]},
})

# Modelos avanzados (metricas en test)
for name, info in results_models.items():
    rows.append({"method": "Default", "model": name, **info["test"]})

# HPO (todas las combinaciones metodo x modelo)
for method, models_dict in results_hpo.items():
    for model_name, info in models_dict.items():
        rows.append({"method": method, "model": model_name, **info["test"]})

# Dim reduction (solo tecnicas con metricas de modelo)
for tech, info in results_dimred.items():
    if "metrics" in info:
        rows.append({"method": f"DimRed-{tech}", "model": "RF-quick", **info["metrics"]})

final_results_df = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
print(final_results_df[["method", "model", "accuracy", "precision", "recall", "f1"]].head(10).to_string(index=False))

# Mejor modelo (excluir dim reduction para produccion)
non_dimred = final_results_df[~final_results_df["method"].str.startswith("DimRed")]
best_row = non_dimred.iloc[0]
best_method = best_row["method"]
best_model_name = best_row["model"]
print(f"\nMejor modelo: {best_method} / {best_model_name} (F1={best_row['f1']:.4f})")

# Recuperar estimador
if best_method == "Baseline":
    best_estimator = nb
elif best_method == "Default":
    best_estimator = results_models[best_model_name]["estimator"]
else:
    best_estimator = results_hpo[best_method][best_model_name]["estimator"]

# Validacion Fase 7
assert len(final_results_df) >= 18, f"Se esperan >=18 filas, hay {len(final_results_df)}"
assert final_results_df["f1"].is_monotonic_decreasing, "DataFrame no ordenado por f1 desc"
assert hasattr(best_estimator, "predict"), "best_estimator sin metodo predict"
print(f"[OK] Fase 7 validada. Total filas: {len(final_results_df)}")


=== Fase 7 — Comparacion final ===
    method        model  accuracy  precision  recall     f1
GridSearch      XGBoost    0.8769     0.7755  0.6875 0.7289
    Optuna      XGBoost    0.8761     0.7716  0.6894 0.7282
       NNI      XGBoost    0.8750     0.7659  0.6926 0.7274
  Hyperopt      XGBoost    0.8743     0.7712  0.6792 0.7223
   Default      XGBoost    0.8719     0.7640  0.6773 0.7181
    Random      XGBoost    0.8716     0.7699  0.6658 0.7141
       NNI RandomForest    0.8684     0.7780  0.6346 0.6990
GridSearch RandomForest    0.8676     0.7736  0.6365 0.6984
    Optuna RandomForest    0.8658     0.7621  0.6435 0.6978
  Hyperopt RandomForest    0.8649     0.7618  0.6384 0.6947

Mejor modelo: GridSearch / XGBoost (F1=0.7289)
[OK] Fase 7 validada. Total filas: 18


---
## Fase 8 — Produccion

In [32]:
from sklearn.base import clone

print("=== Fase 8 - Produccion ===")

# Reconstruir X_raw e y desde df ya limpio (sin fnlwgt, target mapeado)
X_raw_final = df.drop(columns=[TARGET_COL])
y_final     = df[TARGET_COL].values

# Clonar mejor estimador; si es XGBoost, intentar usar GPU
classifier = clone(best_estimator)
if hasattr(classifier, "set_params") and "XGBoost" in type(classifier).__name__:
    try:
        classifier.set_params(device="cuda", tree_method="hist")
        print("XGBoost: GPU activada (device=cuda)")
    except Exception as e:
        print(f"GPU no disponible, usando CPU: {e}")
else:
    print(f"Clasificador: {type(classifier).__name__} (CPU)")

final_pipeline = Pipeline(steps=[
    ("preprocessor", ColumnTransformer(
        transformers=[
            ("num", "passthrough", num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ]
    )),
    ("classifier", classifier),
])

final_pipeline.fit(X_raw_final, y_final)

model_path = os.path.join(MODEL_OUTPUT_DIR, "adult_best_model.joblib")
joblib.dump(final_pipeline, model_path)
print(f"Modelo guardado en: {model_path}")

# Smoke test
loaded      = joblib.load(model_path)
sample_pred = loaded.predict(X_raw_final.head(5))
print("Predicciones smoke test:", sample_pred)
print("Ground truth           :", y_final[:5])

# Validacion Fase 8
import pathlib
assert pathlib.Path(model_path).exists(), "Archivo joblib no encontrado"
assert len(sample_pred) == 5, "Smoke test debe retornar 5 predicciones"
assert set(sample_pred).issubset({0, 1}), "Predicciones fuera de {0,1}"
print("[OK] Fase 8 validada. Pipeline guardado y smoke test OK.")

=== Fase 8 - Produccion ===
Clasificador: XGBClassifier (CPU)


Modelo guardado en: C:\Users\USUARIO1\Documents\IA\Adult\models\adult_best_model.joblib
Predicciones smoke test: [0 0 0 0 1]
Ground truth           : [0 0 0 0 0]
[OK] Fase 8 validada. Pipeline guardado y smoke test OK.
